# ATE (Con / EqCon) and methanol selectivity

Four K-means regimes are fitted in standardized performance space. The labels rank the centroids by the horizontal metric and then by selectivity. In each validation fold, scaling, centroids and calcination means are fitted on the fitting records. The XGBoost one-versus-rest models use catalyst and reaction descriptors to predict regime membership.

In [ ]:
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from IPython.display import display
from sklearn.model_selection import train_test_split

from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

import seaborn as sns

from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier

from concurrent.futures import ThreadPoolExecutor
import optuna
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold
from xgboost import XGBRegressor

from sklearn.metrics import accuracy_score

from sklearn.ensemble import IsolationForest

In [ ]:


input_path = Path("derived") / "dataset.xlsx"  # Path to the input workbook.
output_dir = Path("ATE_results")  # Folder for generated results.
output_dir.mkdir(exist_ok=True)
seed = 0  # Random seed for reproducibility.
dpi = 600  # Resolution in dots per inch.

mpl.rcParams.update({
    "font.family": ["Arial", "DejaVu Sans"],
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.linewidth": 1.0,
    "savefig.facecolor": "white",
})

features = [ 
    "wt% Cu", "wt% Support1", "χ Support1", "wt% Support2", "χ Support2",
    "wt% Support3", "χ Support3", "wt% Promoter", "χ Promoter",
    "Synthesis Method", "Calc T", "Calc D", "H2/CO2", "GHSV", "P", "T",
    "SBET", "SACu", "dCu",
]
point_id = "Data point number"  # Unique experimental-record identifier column.
calc_columns = ["Calc T", "Calc D"]  # Calcination temperature and duration columns.
flag_columns = [f"{column} imputed" for column in calc_columns]  # Columns indicating missing calcination values. Current column name.
data = pd.read_excel(input_path, sheet_name="data", engine="openpyxl")  # Input records for this analysis.
if data[point_id].isna().any() or not data[point_id].is_unique:
    raise ValueError("Data point numbers must be present and unique.")
X = data[features].apply(pd.to_numeric).astype(float)  # Predictor matrix.
if not np.isfinite(X.to_numpy()).all():
    raise ValueError("Encoded model inputs must be finite.")
for column, flag in zip(calc_columns, flag_columns):  # Current column name; Indicator for the current record or column.
    if not data[flag].isin([False, True]).all():
        raise ValueError(f"{flag} must contain Boolean values.")
    X[column] = X[column].mask(data[flag].astype(bool))

train_idx, test_idx = train_test_split(data.index, test_size=0.2, random_state=seed)  # Indices of the development records; Indices of the held-out records.
train_idx, test_idx = pd.Index(train_idx), pd.Index(test_idx)
split = data[[point_id, "Catalyst number", "Reference DOI"]].assign(Split="Test")  # Training and held-out partition assignments.
split.loc[train_idx, "Split"] = "Train"
print(f"Records: {len(data)} | Train: {len(train_idx)} | Test: {len(test_idx)}")


def calcination_fill(X_fit):  # Return rounded calcination means for the training rows.
    columns = X_fit.columns.intersection(calc_columns)  # Columns included in the operation.
    means = X_fit[columns].mean()  # Calculated means for imputation or plotting.
    if means.isna().any():
        raise ValueError("A training fold lacks observed calcination values.")
    return np.floor(means + 0.5)


def save_figure(fig, filename, resolution=dpi):  # Save a figure in the results folder and close it.
    settings = {"pil_kwargs": {"compression": "tiff_lzw"}} if filename.endswith(".tif") else {}  
    fig.savefig(output_dir / filename, dpi=resolution, bbox_inches="tight", **settings)
    plt.show()
    plt.close(fig)


def panel_letter(ax, letter, x=-0.16, y=1.04):  # Place a panel identifier on its axes.
    ax.text(x, y, f"({letter})", transform=ax.transAxes, ha="left", va="bottom",
            fontsize=12, fontweight="bold")


## Approach to equilibrium

ATE uses the equilibrium conversion supplied in the workbook. It is a dimensionless ratio. Values above one are retained and flagged for review.

In [ ]:
equilibrium = pd.to_numeric(data["EqCon"])  # Equilibrium conversion or gas composition for the current case.
if not np.isfinite(equilibrium).all() or equilibrium.le(0).any():
    raise ValueError("EqCon must be positive and finite.")
data["ATE"] = data["Con"] / equilibrium
if not np.allclose(data["ATE"], data["Con/EqCon"], rtol=1e-6, atol=1e-8):
    raise ValueError("The stored Con/EqCon does not agree with Con divided by EqCon.")
print(f"Recorded ATE above 1: {data['ATE'].gt(1).sum()} rows")

## Performance regimes and support composition

In [ ]:

axis = "ATE"  
axis_label = "ATE (Con / EqCon)"  
outcomes = data[[axis, "Sel"]].astype(float)  # Two response variables used for clustering.
if not np.isfinite(outcomes.to_numpy()).all():
    raise ValueError("Clustering requires finite observed outcomes.")
classes = [0, 1, 2, 3]  # Classes.
palette = {0: "#1f77b4", 1: "#d62728", 2: "#ff7f0e", 3: "#7d4d43"}  


def fit_clusters(frame, n_clusters=4):  # Fit feature scaling and K-means on the training outcomes.
    scaler = StandardScaler()  # Predictor or outcome standardization fitted on training data.
    values = scaler.fit_transform(frame)  # Values used in the current calculation.
    with threadpool_limits(limits=1):
        model = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10).fit(values)  # Estimator for the current task.
    return scaler, model


def ranked_clusters(scaler, model):  # Give fitted clusters a stable order based on their outcome centers.
    centers = model.cluster_centers_  
    ordered = np.argsort(centers[:, 0])  
    labels = np.empty(4, dtype=int)  # Group or display labels.
    for side, ids in zip((ordered[:2], ordered[2:]), ((0, 3), (1, 2))): 
        high, low = side[np.argsort(centers[side, 1])[::-1]]  # Upper bound for the displayed values; Lower bound for the displayed values.
        labels[high], labels[low] = ids
    return scaler, model, labels


def cluster_labels(clusterer, frame):  # Assign outcome rows to the fitted and ordered K-means regimes.
    scaler, model, labels = clusterer  # Predictor or outcome standardization fitted on training data; Estimator for the current task; Group or display labels.
    values = np.ascontiguousarray(scaler.transform(frame), dtype=model.cluster_centers_.dtype)  # Values used in the current calculation.
    with threadpool_limits(limits=1):
        raw = model.predict(values)  # Source records before modeling preparation.
    return pd.Series(labels[raw], index=frame.index, name="Cluster")


clusterer = ranked_clusters(*fit_clusters(outcomes.loc[train_idx]))  # Fitted scaler and ordered cluster assignment.
data["Cluster"] = cluster_labels(clusterer, outcomes)
scaler, kmeans, label_map = clusterer  # Predictor or outcome standardization fitted on training data; Fitted K-means model; Label map.
centers = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), index=label_map, columns=outcomes.columns)  
centers = centers.sort_index().rename_axis("Cluster")
cluster_summary = data.groupby("Cluster").agg(  # Record counts and mean outcomes by regime.
    N=(point_id, "size"), Mean_X=(axis, "mean"), Mean_Sel=("Sel", "mean"),
).rename(columns={"Mean_X": f"Mean {axis}", "Mean_Sel": "Mean Sel"})
cluster_counts = pd.crosstab(data["Cluster"], split["Split"]).reindex(classes, fill_value=0)  
diagnostics = []  # Diagnostics.
for k in range(2, 7):  # K.
    transform, candidate = fit_clusters(outcomes.loc[train_idx], n_clusters=k)  
    diagnostics.append({
        "k": k, "Silhouette": silhouette_score(transform.transform(outcomes.loc[train_idx]), candidate.labels_),
        "Inertia": candidate.inertia_,
    })
cluster_diagnostics = pd.DataFrame(diagnostics) 

fig, ax = plt.subplots(figsize=(9, 7), layout="tight")  # Matplotlib figure; Plot axes.
for cid, points in data.groupby("Cluster"):  # Performance regime identifier; Points.
    ax.scatter(points[axis], points["Sel"], color=palette[cid], s=48, edgecolor="black",
               linewidth=0.65, alpha=0.95, label=str(cid), zorder=3)
ax.axvline(outcomes.loc[train_idx, axis].median(), linestyle="--", color="0.65", linewidth=0.8)
ax.axhline(outcomes.loc[train_idx, "Sel"].median(), linestyle="--", color="0.65", linewidth=0.8)
ax.set(xlabel=axis_label, ylabel="Selectivity (%)", title="Catalyst Performance Quadrants")
ax.set_axisbelow(True)
ax.grid(linestyle=":", alpha=0.55)
ax.legend(title="Quadrant ID", frameon=True, loc="upper right")
save_figure(fig, "performance_quadrants.tif")  # Save Fig. 7, the ATE–selectivity K-means regimes.
display(cluster_summary.join(cluster_counts).round(3))

In [ ]:

absent = {"0", "0.0", "0.00", "", "none", "nan", "<na>"}  
support_columns = ["Support1", "Support2", "Support3"] 
support = data[support_columns].fillna("").astype(str)  
data["Support combination"] = support.apply(
    lambda row: "+".join(s.strip() for s in row if s.strip().lower() not in absent) or "Bare Cu",
    axis=1,
)
support_minimum = 5  
counts = data["Support combination"].value_counts() 
kept = data[data["Support combination"].isin(counts[counts.ge(support_minimum)].index)]  
support_counts = pd.crosstab(kept["Cluster"], kept["Support combination"]).reindex(classes, fill_value=0)  # Support counts.
hit_rates = 100 * support_counts.div(support_counts.sum(axis=0), axis=1)  
hit_rates = hit_rates.loc[:, hit_rates.loc[1].sort_values(ascending=False).index]
top_supports = hit_rates.loc[1].head(5).rename("Cluster 1 share (%)").to_frame()  # Top supports.
top_supports["Total records"] = counts
top_supports["Cluster 1 records"] = support_counts.loc[1]
top_supports[f"Mean {axis} in Cluster 1"] = data.loc[data["Cluster"].eq(1)].groupby("Support combination")[axis].mean()

fig, ax = plt.subplots(figsize=(max(9, 0.55 * hit_rates.shape[1] + 2), 5.2), layout="constrained")
sns.heatmap(hit_rates, annot=True, fmt=".1f", cmap="YlOrRd", vmin=0, vmax=100, linewidths=0.35,
            ax=ax, cbar_kws={"label": "Share of records (%)"})
ax.set(xlabel="Support combination (sorted by Cluster 1 share)", ylabel="Cluster ID",
       title="Support Hit Rates by Performance Cluster")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
save_figure(fig, "Support_Hit_Rates.tif")
display(top_supports.round(3))

## Classifier validation and SHAP

In [ ]:
def tree_explanation(model, X_fit, X_explain):  # Calculate TreeSHAP values for the requested feature rows.
    background = shap.sample(X_fit, min(100, len(X_fit)), random_state=seed)  
    explainer = shap.TreeExplainer(model, data=background, feature_perturbation="interventional")  # TreeSHAP explainer.
    explanation = explainer(X_explain, check_additivity=False)  # TreeSHAP values and their feature data.
    if explanation.values.shape != X_explain.shape:
        raise ValueError("Expected one SHAP value for every sample and feature.")
    return explanation


def shap_panels(explanation, ax_swarm, ax_bar, decimals=3, other_label="Sum of others"):  # Plot SHAP distributions and rank features by mean absolute SHAP.
    mean_abs = np.abs(explanation.values).mean(axis=0)  
    ranked = np.argsort(-mean_abs) 
    top = ranked[:8]  # Highest-ranked entries selected for display.
    top_names = [explanation.feature_names[i] for i in top]  
    subset = shap.Explanation(  
        values=explanation.values[:, top], data=explanation.data[:, top],
        base_values=explanation.base_values, feature_names=top_names,
    )
    plt.sca(ax_swarm)
    np.random.seed(seed)
    shap.plots.beeswarm(subset, show=False, plot_size=None, color=plt.get_cmap("coolwarm"))
    ax_swarm.set_xlabel("SHAP value")
    ax_swarm.spines["bottom"].set_linewidth(1.2)
    ax_swarm.grid(axis="x", linestyle="--", alpha=0.25)
    values = np.r_[mean_abs[top], mean_abs[ranked[8:]].sum()]  # Values used in the current calculation.
    bars = ax_bar.barh(  
        top_names + [other_label], values, color=["#1f77b4"] * len(top) + ["#d9d9d9"],
        edgecolor="black", linewidth=0.7, zorder=2,
    )
    ax_bar.invert_yaxis()
    ax_bar.set_xlabel("mean(|SHAP value|)")
    ax_bar.bar_label(bars, fmt=f"%.{decimals}f", padding=3, fontsize=9, fontweight="bold")
    ax_bar.set_xlim(0, max(values.max() * 1.35, 1e-9))
    ax_bar.grid(axis="x", linestyle="--", alpha=0.25)
    return pd.Series(mean_abs, index=explanation.feature_names, name="Mean |SHAP|").sort_values(ascending=False)

In [ ]:

clf_params = {  # Fixed options for the cluster classifiers.
    "n_estimators": 100, "max_depth": 5, "learning_rate": 0.05, "random_state": seed,
    "eval_metric": "logloss", "tree_method": "hist", "n_jobs": 1,
}
X_train = X.loc[train_idx]  # Development-set predictors.
fill = calcination_fill(X_train)  # Rounded imputation values from the fitting data.
X_train = X_train.fillna(fill)
X_test = X.loc[test_idx].fillna(fill)  # Held-out test predictors.
labels = data["Cluster"]  # Group or display labels.
stratified = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)  
cv_sets, stability = [], []  # Prepared cross-validation partitions; Stability.
for fold, (i, j) in enumerate(stratified.split(X_train, labels.loc[train_idx]), start=1):  # Cross-validation fold; Loop position; Second loop position.
    idx_train, idx_val = train_idx[i], train_idx[j]  # Training-side indices for the current fold; Validation-side indices for the current fold.
    cv_clusters = ranked_clusters(*fit_clusters(outcomes.loc[idx_train]))  # Cross-validation clusters.
    cv_train_labels = cluster_labels(cv_clusters, outcomes.loc[idx_train])  # Cross-validation train labels.
    cv_val_labels = cluster_labels(cv_clusters, outcomes.loc[idx_val])  # Cross-validation val labels.
    fold_fill = calcination_fill(X.loc[idx_train])  # Imputation values from this training fold.
    cv_sets.append((
        fold, X.loc[idx_train].fillna(fold_fill), cv_train_labels, X.loc[idx_val].fillna(fold_fill), cv_val_labels,
    ))
    stability.append({
        "Fold": fold, "N": len(idx_val),
        "ARI vs full training clustering": adjusted_rand_score(labels.loc[idx_val], cv_val_labels),
    })


def classification_scores(actual, prob):  # Compute binary scores and confusion counts for one regime.
    pred = prob >= 0.5  # Model predictions.
    tn, fp, fn, tp = confusion_matrix(actual, pred, labels=[0, 1]).ravel()  # True negatives; False positives; False negatives; True positives.
    return {
        "N": len(actual), "Positive": int(actual.sum()),
        "ROC AUC": roc_auc_score(actual, prob) if actual.nunique() == 2 else np.nan,
        "Average precision": average_precision_score(actual, prob) if actual.any() else np.nan,
        "Balanced accuracy": balanced_accuracy_score(actual, pred),
        "Precision": precision_score(actual, pred, zero_division=0),
        "Recall": recall_score(actual, pred, zero_division=0),
        "F1": f1_score(actual, pred, zero_division=0), "TN": tn, "FP": fp, "FN": fn, "TP": tp,
    }


explanations, rankings = {}, []  
validation_scores, test_scores = [], []  
predictions = split.join(outcomes).assign(Cluster=labels, Support=data["Support combination"])  # Record-level predictions.
for cid in classes:
    for fold, X_tr, y_tr, X_va, y_va in cv_sets:  # Training-fold predictors; Training-fold responses; Validation-fold predictors; Validation-fold responses.
        model = XGBClassifier(**clf_params)  # Estimator for the current task.
        model.fit(X_tr, y_tr.eq(cid).astype(int))
        prob = model.predict_proba(X_va)[:, 1]  # Predicted positive-class probability.
        validation_scores.append({"Cluster": cid, "Fold": fold, **classification_scores(y_va.eq(cid), prob)})
    model = XGBClassifier(**clf_params)
    model.fit(X_train, labels.loc[train_idx].eq(cid).astype(int))
    prob = model.predict_proba(X_test)[:, 1]
    test_scores.append({"Cluster": cid, **classification_scores(labels.loc[test_idx].eq(cid), prob)})
    predictions.loc[test_idx, f"Probability cluster {cid}"] = prob
    explanations[cid] = tree_explanation(model, X_train, X_train)

cv_metrics = pd.DataFrame(validation_scores)  
cv_summary = cv_metrics.groupby("Cluster")[["ROC AUC", "Average precision", "Balanced accuracy", "F1"]].agg(["mean", "std"])  
test_metrics = pd.DataFrame(test_scores)  
fig, axes = plt.subplots(2, 2, figsize=(8, 7), layout="constrained")  # Plot axes arranged by panel.
for ax, row in zip(axes.flat, test_scores):  
    matrix = [[row["TN"], row["FP"]], [row["FN"], row["TP"]]]  
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", cbar=False, square=True, ax=ax)
    ax.set(title=f"Cluster {row['Cluster']}", xlabel="Predicted membership", ylabel="Observed membership")
save_figure(fig, "Test_Confusion_Matrices.tif")  # Save Fig. S7, the held-out confusion matrices for clusters 0–3 in reading order.
display(test_metrics.round(3))

In [ ]:
def draw_cluster_shap(ax0, ax1, cid):  # Draw the SHAP beeswarm and importance bars for one regime.
    ranking = shap_panels(explanations[cid], ax0, ax1)  
    panel_letter(ax0, chr(97 + 2*cid), x=-0.48, y=1.01)
    panel_letter(ax1, chr(98 + 2*cid), x=-0.58, y=1.01)
    ax0.text(1.16, 1.12, f"SHAP analysis for Cluster {cid} Identification",
             transform=ax0.transAxes, ha="center", va="bottom", fontsize=11, fontweight="bold")
    return ranking


for cid in classes:
    fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(8.5, 4.8), gridspec_kw={"width_ratios": [1.24, 1]})  # First plot axes; Second plot axes.
    fig.subplots_adjust(left=0.20, right=0.96, wspace=0.84, top=0.82, bottom=0.16)
    ranking = draw_cluster_shap(ax0, ax1, cid)  
    rankings.append(ranking.rename_axis("Feature").reset_index().assign(Cluster=cid))
    save_figure(fig, f"Classification_SHAP_Cluster_{cid}.tif")  # Save the corresponding pair of panels in Fig. 8 (a,b for cluster 0 through g,h for cluster 3).

fig, axes = plt.subplots(4, 2, figsize=(8.5, 19.2), gridspec_kw={"width_ratios": [1.24, 1]})
fig.subplots_adjust(left=0.20, right=0.96, wspace=0.84, hspace=0.60, top=0.95, bottom=0.04)
for cid, (ax0, ax1) in zip(classes, axes):
    draw_cluster_shap(ax0, ax1, cid)
save_figure(fig, "SHAP_All_Clusters_Vertical.tif", resolution=300)  # Save Fig. 8a–h, paired SHAP panels for ATE–selectivity clusters 0–3.

pairs = [("T", "wt% Support1"), ("wt% Support1", "T"), ("GHSV", "T")]  
fig, axes = plt.subplots(1, 3, figsize=(18, 5), layout="constrained")
for ax, (feature, color_feature) in zip(axes, pairs):  
    shap.dependence_plot(
        feature, explanations[1].values, X_train, interaction_index=color_feature,
        ax=ax, cmap="coolwarm", show=False, alpha=0.8, dot_size=12,
    )
    ax.set_title(f"SHAP Dependence: {feature}")
    ax.grid(linestyle="--", alpha=0.25)
save_figure(fig, "Dependence_SHAP_Cluster_1_Interactions.tif")

regime_tables = {  # Worksheet names and clustering or regression results to export.
    "predictions": predictions, "cluster_summary": cluster_summary.join(cluster_counts),
    "cluster_centers": centers, "cluster_diagnostics": cluster_diagnostics,
    "cluster_stability": pd.DataFrame(stability), "support_counts": support_counts,
    "support_hit_rates": hit_rates, "top_supports": top_supports,
    "classifier_cv": cv_metrics, "classifier_cv_summary": cv_summary,
    "classifier_test": test_metrics, "classifier_SHAP": pd.concat(rankings, ignore_index=True),  # Prepare the ATE–selectivity rows of Table S4 for export.
}

## ATE and selectivity regression

The screening regressors use the original 16-feature set. Optuna uses five folds from the training split. After recording test scores, models fitted on all literature rows are used for virtual screening.

In [ ]:


optuna.logging.set_verbosity(optuna.logging.WARNING)
n_splits = 5  # Number of cross-validation folds.
cv_workers = 5  # Cross-validation workers.
folds = list(KFold(n_splits=n_splits, shuffle=True, random_state=seed).split(train_idx))  # Cross-validation splits.
model_defaults = {"objective": "reg:squarederror", "tree_method": "hist", "random_state": seed, "n_jobs": 1}  # Fixed estimator options.


def prediction(model, values, upper=None):  # Predict responses and enforce the configured prediction bounds.
    return np.clip(model.predict(values), 0, upper)


def regression_scores(y, pred):  # Compute regression performance from paired observed and predicted values.
    return {"R2": r2_score(y, pred), "RMSE": np.sqrt(mean_squared_error(y, pred)), "MAE": mean_absolute_error(y, pred)}


def fit_regression(target, columns, trials, upper=None, early_stop=False):  # Tune and fit an XGBoost regressor using the training folds.
    y = data[target].astype(float)  # Observed response.
    if not np.isfinite(y.to_numpy()).all():
        raise ValueError(f"{target} must contain finite observations.")
    X_fit = X.loc[train_idx, columns]  # Predictors used to fit the estimator.
    cv_sets = []  # Prepared cross-validation partitions.
    for fold, (i, j) in enumerate(folds, start=1):  # Cross-validation fold; Loop position; Second loop position.
        X_tr, X_va = X_fit.iloc[i], X_fit.iloc[j]  # Training-fold predictors; Validation-fold predictors.
        fill = calcination_fill(X_tr)  # Rounded imputation values from the fitting data.
        cv_sets.append((fold, X_tr.fillna(fill), y.loc[X_tr.index], X_va.fillna(fill), y.loc[X_va.index]))

    def score_fold(params, fold_data):  # Fit and evaluate the regressor for one validation split.
        fold, X_tr, y_tr, X_va, y_va = fold_data  # Cross-validation fold; Training-fold predictors; Training-fold responses; Validation-fold predictors; Validation-fold responses.
        if early_stop:
            model = XGBRegressor(**model_defaults, **params, n_estimators=1000, early_stopping_rounds=20)  # Estimator for the current task.
            model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
            rounds = model.best_iteration + 1  # Number of boosting iterations.
        else:
            model = XGBRegressor(**model_defaults, **params)
            model.fit(X_tr, y_tr)
            rounds = params["n_estimators"]
        return {
            "Fold": fold, "Rounds": rounds, **regression_scores(y_va, prediction(model, X_va, upper)),
            "Train RMSE": regression_scores(y_tr, prediction(model, X_tr, upper))["RMSE"],
        }

    with ThreadPoolExecutor(max_workers=cv_workers) as pool:  
        def objective(trial):  # Return cross-validation error for the current Optuna trial.
            if early_stop:
                params = {  # Estimator hyperparameters.
                    "max_depth": trial.suggest_int("max_depth", 2, 5),
                    "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                    "min_child_weight": trial.suggest_int("min_child_weight", 10, 30),
                    "subsample": trial.suggest_float("subsample", 0.5, 0.85),
                    "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 0.85),
                    "reg_alpha": trial.suggest_float("reg_alpha", 1e-2, 10, log=True),
                    "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 50, log=True),
                    "gamma": trial.suggest_float("gamma", 0, 5),
                }
            else:
                params = {
                    "n_estimators": trial.suggest_int("n_estimators", 50, 300),
                    "max_depth": trial.suggest_int("max_depth", 3, 10),
                    "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
                    "subsample": trial.suggest_float("subsample", 0.6, 1.0),
                    "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
                }
            scores = list(pool.map(lambda s: score_fold(params, s), cv_sets))  # Scores for the current evaluation.
            trial.set_user_attr("fold_scores", scores)
            return float(np.mean([row["RMSE"] for row in scores])) 

        study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=seed))  # Optuna hyperparameter search.
        print(f"Searching {target}: {trials} trials × {n_splits} folds", flush=True)
        study.optimize(objective, n_trials=trials)

    params = study.best_params.copy()  # Estimator hyperparameters.
    fold_scores = pd.DataFrame(study.best_trial.user_attrs["fold_scores"]).assign(Target=target)  
    if early_stop:
        params["n_estimators"] = max(1, int(1.1 * fold_scores["Rounds"].mean()))
    fill = calcination_fill(X_fit)
    fitted_X = X_fit.fillna(fill)  
    test_X = X.loc[test_idx, columns].fillna(fill)  
    model = XGBRegressor(**model_defaults, **params)  # Estimator for the current task.
    model.fit(fitted_X, y.loc[train_idx])

    predicted = pd.Series(index=data.index, dtype=float, name="Predicted")  # Predicted response values.
    metrics = []  # Performance metrics.
    for subset, values in [("Train", fitted_X), ("Test", test_X)]:  # Subset; Values used in the current calculation.
        pred = prediction(model, values, upper)  # Model predictions.
        predicted.loc[values.index] = pred
        metrics.append({"Target": target, "Split": subset, "N": len(pred), **regression_scores(y.loc[values.index], pred)})
    print(f"{target} | trial {study.best_trial.number} | CV RMSE {study.best_value:.4f}", flush=True)
    return model, fitted_X, test_X, fill, params, predicted, pd.DataFrame(metrics), fold_scores, study


def parity_panel(ax, datasets, label, mean_cv_r2=None):  # Plot observed and predicted values on equal axes.
    color_map = {"Train": "#2a9d8f", "Test": "#f4a261"}  
    values = np.concatenate([v for _, actual, pred in datasets for v in (actual, pred)])  # Values used in the current calculation. Observed values; Model predictions. Current array or scalar.
    pad = 0.05 * max(np.ptp(values), 1e-9)  # Pad.
    limits = (values.min() - pad, values.max() + pad)  # Limits.
    annotations = []  # Annotations.
    for zorder, (subset, actual, pred) in enumerate(datasets, start=2):  
        score = regression_scores(actual, pred)  # Score.
        ax.scatter(actual, pred, s=40, color=color_map[subset], edgecolor="black", linewidth=0.8,
                   alpha=0.7 if subset == "Train" else 0.8,
                   label=subset, zorder=zorder)
        annotations.append(f"{subset} $R^2$ = {score['R2']:.2f}")
    if mean_cv_r2 is not None:
        annotations.append(f"Mean $R^2$ = {mean_cv_r2:.2f}")
    ax.plot(limits, limits, linestyle="--", dashes=(5, 4), color="black", linewidth=1.5, zorder=1)
    ax.set(xlim=limits, ylim=limits, xlabel=f"Actual {label}", ylabel=f"Predicted {label}")
    ax.set_aspect("equal", adjustable="box")
    for spine in ax.spines.values():  # Spine.
        spine.set_visible(True)
        spine.set_linewidth(1.5)
    ax.tick_params(labelsize=11, direction="in", length=6, width=1.5)
    ax.legend(frameon=False, loc="upper left", fontsize=12)
    ax.text(0.95, 0.05, "\n".join(annotations), transform=ax.transAxes, ha="right", va="bottom",
            multialignment="left", fontsize=12, fontweight="bold",
            bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "edgecolor": "none", "alpha": 0.8},
            zorder=4)

In [ ]:
ate_trials = 30  
screening_features = [column for column in features if column not in {"Calc T", "Calc D", "Synthesis Method"}]  # Screening features. Current column name.
X_screen = X[screening_features]  
target_limits = {"ATE": None, "Sel": 100}  
target_labels = {"ATE": "ATE (Con / EqCon)", "Sel": "Methanol selectivity (%)"} 
screening_models = {}  
regression_metrics, regression_folds, regression_params, regression_trials = [], [], [], []  
fig, axes = plt.subplots(1, 2, figsize=(11, 5), layout="constrained")
for ax, (target, upper) in zip(axes, target_limits.items()):  # Target; Upper bound on predicted values.
    _, _, _, _, params, predicted, metrics, fold_scores, study = fit_regression(  # Estimator hyperparameters; Predicted response values; Performance metrics; Fold scores; Optuna hyperparameter search.
        target, screening_features, ate_trials, upper,
    )
    regression_metrics.append(metrics)
    regression_folds.append(fold_scores)
    regression_params.append({"Target": target, **params})
    regression_trials.append(study.trials_dataframe(attrs=("number", "value", "params", "state")).assign(Target=target))
    predictions[f"Regressed {target}"] = predicted
    predictions[f"{target} residual"] = predicted - data[target]
    parity_panel(ax, [
        ("Train", data.loc[train_idx, target], predicted.loc[train_idx]),
        ("Test", data.loc[test_idx, target], predicted.loc[test_idx]),
    ], target_labels[target])
    screening_models[target] = XGBRegressor(**model_defaults, **params).fit(X_screen, data[target])
predictions["ATE above 1"] = predictions["ATE"].gt(1)
save_figure(fig, "ATE_Sel_Regression.tif")
display(pd.concat(regression_metrics).round(4))

## Held-out cascaded regime prediction

The regressors predict ATE and selectivity for the held-out records. The fitted clustering model then assigns the predicted and observed outcomes to regimes.

In [ ]:

predictions["Regressed cluster"] = cluster_labels(
    clusterer, predictions[["Regressed ATE", "Regressed Sel"]].set_axis([axis, "Sel"], axis=1),
)
actual = predictions.loc[test_idx, "Cluster"]  # Observed values.
predicted = predictions.loc[test_idx, "Regressed cluster"]
matrix = confusion_matrix(actual, predicted, labels=classes)
regime_tables["cascade_test_matrix"] = pd.DataFrame(
    matrix, columns=[f"Predicted {cid}" for cid in classes],  # Performance regime identifier.
).assign(Observed=classes)
cascade_metrics = pd.DataFrame([{"Split": "Test", "N": len(actual), "Accuracy": accuracy_score(actual, predicted)}])  # Cascade metrics.
display(cascade_metrics.round(4))


## Virtual screening of catalyst–condition combinations

Catalyst composition and characterization are sampled jointly from observed literature rows. T, P, H₂/CO₂ and GHSV are varied within their 5th–95th percentile intervals. The original Isolation Forest and ATE constraints screen the resulting combinations before Pareto ranking. Candidates are model predictions.

In [ ]:

screening_trials = 2000  # Screening trials.
conditions = ["H2/CO2", "GHSV", "P", "T"]  # Reaction conditions used in the analysis.
composition_features = X_screen.columns.difference(conditions, sort=False)  
mass_columns = ["wt% Cu", "wt% Support1", "wt% Support2", "wt% Support3", "wt% Promoter"]  
support_codes = ["χ Support1", "χ Support2", "χ Support3"] 
quantiles = X_screen.quantile([0.05, 0.95]).T.set_axis(["Lower (5%)", "Upper (95%)"], axis=1) 
valid_mass = X_screen[mass_columns].ge(0).all(axis=1) & X_screen[mass_columns].sum(axis=1).le(100.01) 
support_list = X_screen[support_codes].replace(0, np.nan) 
unique_supports = support_list.nunique(axis=1).eq(support_list.count(axis=1))  
recipes = X_screen.loc[valid_mass & unique_supports, composition_features].drop_duplicates()  
recipe_ids = list(recipes.index)  # Recipe ids.
if not recipe_ids:
    raise ValueError("No observed catalyst composition meets the screening constraints.")
isolation = IsolationForest(n_estimators=100, contamination=0.01, random_state=42, n_jobs=1).fit(X_screen)  
literature_scores = isolation.decision_function(X_screen)  
tolerance = literature_scores.min() - 0.05 * np.ptp(literature_scores)  


def screen_candidate(trial):  # Score a virtual candidate with the fitted screening models.
    recipe = trial.suggest_categorical("Recipe row", recipe_ids)  
    values = recipes.loc[recipe].to_dict()  # Values used in the current calculation.
    for feature in conditions:  
        low, high = quantiles.loc[feature]  # Lower bound for the displayed values; Upper bound for the displayed values.
        values[feature] = trial.suggest_float(feature, low, high)
    sample = pd.DataFrame([values], columns=screening_features)  
    score = float(isolation.decision_function(sample)[0])  
    if score < tolerance:
        raise optuna.TrialPruned()
    ate = float(prediction(screening_models["ATE"], sample)[0])  # Measured conversion divided by equilibrium conversion.
    if ate >= 1:
        raise optuna.TrialPruned()
    selectivity = float(prediction(screening_models["Sel"], sample, 100)[0])  
    trial.set_user_attr("Isolation score", score)
    return ate, selectivity


search = optuna.create_study(directions=["maximize", "maximize"], sampler=optuna.samplers.TPESampler(seed=seed))  
print(f"Screening {screening_trials} catalyst–condition combinations", flush=True)
search.optimize(screen_candidate, n_trials=screening_trials)
candidate_rows = []  # Candidate rows.
for trial in search.best_trials:  
    recipe = trial.params["Recipe row"]  
    candidate_rows.append({
        "Trial": trial.number, "Predicted ATE": trial.values[0], "Predicted Sel": trial.values[1],
        "Isolation score": trial.user_attrs["Isolation score"], "Source data point": data.at[recipe, point_id],
        **recipes.loc[recipe].to_dict(),
        **data.loc[recipe, ["Support1", "Support2", "Support3", "Promoter"]].to_dict(),
        **{feature: trial.params[feature] for feature in conditions},  
    })
candidates = pd.DataFrame(candidate_rows, columns=[  
    "Trial", "Predicted ATE", "Predicted Sel", "Isolation score", "Source data point", *composition_features,
    "Support1", "Support2", "Support3", "Promoter", *conditions,
]).sort_values("Predicted ATE", ascending=False)
all_trials = search.trials_dataframe(attrs=("number", "values", "params", "state")) 

fig, ax = plt.subplots(figsize=(7, 4.5), layout="tight")
ax.hist(literature_scores, bins=40, color="skyblue", edgecolor="black", alpha=0.7)
ax.axvline(literature_scores.min(), color="#f4a261", linewidth=1.5, label="Minimum observed score")
ax.axvline(tolerance, color="#D62728", linestyle="--", label=f"Screening threshold ({tolerance:.3f})")
ax.set(xlabel="Isolation Forest decision score", ylabel="Number of records")
ax.legend(frameon=False)
ax.grid(axis="y", linestyle="--", alpha=0.25)
save_figure(fig, "Isolation_Forest_Scores.tif")

fig, ax = plt.subplots(figsize=(6, 4.5))
completed = [trial.values for trial in search.trials if trial.state == optuna.trial.TrialState.COMPLETE]  # Completed. Current Optuna trial.
if completed:
    ax.scatter(*np.asarray(completed).T, color="#82B5D2", s=12, alpha=0.55, label="Accepted trials")
if not candidates.empty:
    ax.scatter(candidates["Predicted ATE"], candidates["Predicted Sel"], color="#D62728", s=26, label="Pareto front")
ax.set(xlabel="Predicted ATE (Con / EqCon)", ylabel="Predicted methanol selectivity (%)")
if completed:
    ax.legend(frameon=False)
save_figure(fig, "Virtual_Screening_Pareto.tif")
print(f"Accepted: {len(completed)} | Pareto candidates: {len(candidates)}")
display(candidates.head(5))

## Export

In [ ]:
regression_cv = pd.concat(regression_folds, ignore_index=True)  # Regression cross-validation.
regime_tables.update({
    "predictions": predictions, "regression_metrics": pd.concat(regression_metrics, ignore_index=True),
    "regression_cv": regression_cv,
    "regression_params": pd.DataFrame(regression_params),
    "regression_trials": pd.concat(regression_trials, ignore_index=True),
    "cascade_metrics": cascade_metrics,
    "screening_bounds": quantiles.rename_axis("Feature").reset_index(),
    "screening_candidates": candidates, "screening_trials": all_trials,
})

In [ ]:
with pd.ExcelWriter(output_dir / "ATE_results.xlsx", engine="openpyxl") as writer:  # Excel writer for the result workbook.
    indexed = {"cluster_summary", "cluster_centers", "support_counts", "support_hit_rates", "top_supports", "classifier_cv_summary"}  # Indexed.
    for sheet, table in regime_tables.items():  # Result workbook worksheet name; Result table.
        table.to_excel(writer, sheet_name=sheet, index=sheet in indexed)  # Export the ATE–selectivity rows of Table S4 in the classifier_test sheet.
print(f"Saved tables and figures to {output_dir.resolve()}")